# Bitcoin Master — Final Artifact-Driven Workflow

## Role
Safe-mode interface to the final ten-model Bitcoin experiment.

## Inputs
Frozen point vectors and corrected derived Bitcoin evidence.

## Outputs
Complete final synthesis without generation.

## Depends On
01–12.

## Authoritative Status
MASTER / ORCHESTRATION

## What This Notebook Does Not Do
Default Run All does not train, load external checkpoints, regenerate forecasts, or modify artifacts.


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.bitcoin_pipeline import *
RUN_GENERATION = False
PROMOTE_TO_AUTHORITATIVE = False


In [2]:
daily,target=load_bitcoin_target(ROOT); train,test=canonical_split(target); v=load_validated_forecasts(ROOT); forecasts=forecast_series(v,target); pd.DataFrame({'Daily rows':[len(target)],'Train':[len(train)],'Test':[len(test)],'Test start':[test.index.min()],'Test end':[test.index.max()]})

,Daily rows,Train,Test,Test start,Test end
0,5302,4241,1061,2023-08-12 00:00:00+00:00,2026-07-07 00:00:00+00:00


## Rolling one-step protocol
At target date t, only observations strictly before t are available. Actual t is revealed only after the forecast is recorded.

In [3]:
pd.DataFrame({'Model':FINAL_MODEL_ORDER,'Context / update policy':[CONTEXT_POLICY[m] for m in FINAL_MODEL_ORDER]})

,Model,Context / update policy
0,Naive,t-1 only
1,Simple Exponential Smoothing — Rolling One-Step,last 128 prices; daily refit
2,Additive-Trend Exponential Smoothing,last 128 prices; daily refit
3,ARIMA Rolling One-Step,initial 128 returns plus daily state update
4,7-Day Moving Average,last 7 prices; daily update
5,Prophet — 30-Day Periodic Refit,128 prices at refit origin; refit every 30 tar...
6,Persistence-Enhanced Log-Return LSTM,last 30 returns; fixed trained model
7,Persistence-Enhanced Log-Return Transformer,last 128 returns; fixed trained model
8,Chronos-Bolt-Tiny,last 128 prices; zero-shot; horizon 1
9,TimesFM,last 128 prices; zero-shot; horizon 1


In [4]:
metrics=pd.read_csv(ROOT/'results'/'bitcoin_point_forecast_metrics_v2.csv'); metrics.sort_values('RMSE')

,Model,MAE,RMSE,MAPE,sMAPE,MASE,N,Relative MAE vs Naive
0,Naive,1290.353242,1853.624774,1.742747,1.744142,4.575633,1061,1.000000
1,Simple Exponential Smoothing — Rolling One-Step,1290.358684,1855.731424,1.742685,1.743871,4.575652,1061,1.000004
3,ARIMA Rolling One-Step,1299.874638,1866.302859,1.754004,1.754209,4.609396,1061,1.007379
2,Additive-Trend Exponential Smoothing,1308.541314,1871.702185,1.763640,1.763424,4.640128,1061,1.014095
6,Persistence-Enhanced Log-Return LSTM,1321.365311,1881.091190,1.783956,1.791645,4.685603,1061,1.024034
9,TimesFM,1349.946786,1924.199337,1.823179,1.823895,4.786953,1061,1.046184
8,Chronos-Bolt-Tiny,1424.025828,1994.007926,1.934509,1.928782,5.049640,1061,1.103594
7,Persistence-Enhanced Log-Return Transformer,2019.366342,2559.749810,2.710936,2.759271,7.160736,1061,1.564972
4,7-Day Moving Average,2209.776153,2999.605073,3.021810,3.024208,7.835935,1061,1.712536
5,Prophet — 30-Day Periodic Refit,8195.262862,10781.162873,11.199767,11.287185,29.060658,1061,6.351178


In [5]:
pd.read_csv(ROOT/'results'/'bitcoin_regime_robustness_training_defined.csv').groupby(['Regime','Model']).RMSE.first().unstack(0)

Regime,High Volatility,Low Volatility,Major Downward Movement,Major Upward Movement
Model,,,,
7-Day Moving Average,4170.859266,2650.216436,4183.552465,4035.584274
ARIMA Rolling One-Step,2909.849599,1552.531433,3062.506680,3301.093740
Additive-Trend Exponential Smoothing,3007.655010,1543.573642,3079.387401,3313.462327
Chronos-Bolt-Tiny,3082.246687,1643.695042,3376.049614,3205.191045
Naive,2991.583302,1529.214419,3004.313232,3370.396446
Persistence-Enhanced Log-Return LSTM,3011.953668,1553.524086,2741.571195,3615.754736
Persistence-Enhanced Log-Return Transformer,3404.246296,2384.555004,1574.259557,4900.648108
Prophet — 30-Day Periodic Refit,14327.143627,9909.703453,11584.230460,10921.356764
Simple Exponential Smoothing — Rolling One-Step,2975.692511,1540.742383,3013.243992,3352.705268


In [6]:
pd.read_csv(ROOT/'results'/'bitcoin_temporal_stability.csv').groupby(['Segment','Model']).RMSE.first().unstack(0)

Segment,Earlier,Later,Middle
Model,,,
7-Day Moving Average,2246.462544,3228.992440,3394.693166
ARIMA Rolling One-Step,1402.661622,1964.653266,2150.116588
Additive-Trend Exponential Smoothing,1428.902880,1945.715710,2164.031488
Chronos-Bolt-Tiny,1531.946847,2128.869810,2247.404213
Naive,1408.715265,1931.860167,2142.906579
Persistence-Enhanced Log-Return LSTM,1429.052390,1948.165364,2186.023496
Persistence-Enhanced Log-Return Transformer,1863.520009,2685.069427,2996.082331
Prophet — 30-Day Periodic Refit,8067.194792,12653.359424,11119.227496
Simple Exponential Smoothing — Rolling One-Step,1410.171763,1937.529135,2142.309573


In [7]:
pd.read_csv(ROOT/'results'/'bitcoin_uncertainty_evidence_v2.csv')

,Model,Uncertainty Evidence Type,Native,Calibrated,Calibration Source,Nominal Level,Coverage,Average Width,Coverage Error,Evidence Status
0,Naive,Validation-Residual Empirical Interval,False,True,"Final 1,061 training dates",0.80,0.656927,2766.580000,0.143073,Available
1,Naive,Validation-Residual Empirical Interval,False,True,"Final 1,061 training dates",0.95,0.903864,5985.980000,0.046136,Available
2,7-Day Moving Average,Validation-Residual Empirical Interval,False,True,"Final 1,061 training dates",0.80,0.667295,5119.605714,0.132705,Available
3,7-Day Moving Average,Validation-Residual Empirical Interval,False,True,"Final 1,061 training dates",0.95,0.909519,10181.151429,0.040481,Available
4,ARIMA Rolling One-Step,Validation-Residual Empirical Interval,False,True,"Final 1,061 training dates",0.80,0.662582,2804.023415,0.137418,Available
5,ARIMA Rolling One-Step,Validation-Residual Empirical Interval,False,True,"Final 1,061 training dates",0.95,0.899152,6024.447789,0.050848,Available
6,Simple Exponential Smoothing — Rolling One-Step,Validation-Residual Empirical Interval,False,True,"Final 1,061 training dates",0.80,0.669180,2841.120081,0.130820,Available
7,Simple Exponential Smoothing — Rolling One-Step,Validation-Residual Empirical Interval,False,True,"Final 1,061 training dates",0.95,0.898209,5900.360003,0.051791,Available
8,Additive-Trend Exponential Smoothing,Validation-Residual Empirical Interval,False,True,"Final 1,061 training dates",0.80,0.662582,2803.806369,0.137418,Available
9,Additive-Trend Exponential Smoothing,Validation-Residual Empirical Interval,False,True,"Final 1,061 training dates",0.95,0.902922,5957.324953,0.047078,Available


In [8]:
dm=pd.read_csv(ROOT/'results'/'bitcoin_dm_pairwise_results_hac_holm.csv'); dm[['Model A','Model B','Raw p-value','Holm-adjusted p-value','Significant Holm?','Lower-RMSE model']]

,Model A,Model B,Raw p-value,Holm-adjusted p-value,Significant Holm?,Lower-RMSE model
0,Naive,Simple Exponential Smoothing — Rolling One-Step,6.915669e-01,1.000000e+00,False,Naive
1,Naive,Additive-Trend Exponential Smoothing,3.321510e-02,2.934753e-01,False,Naive
2,Naive,ARIMA Rolling One-Step,2.520136e-01,1.000000e+00,False,Naive
3,Naive,7-Day Moving Average,3.645206e-14,9.842055e-13,True,Naive
4,Naive,Prophet — 30-Day Periodic Refit,1.525571e-18,5.918893e-17,True,Naive
5,Naive,Persistence-Enhanced Log-Return LSTM,6.560186e-03,7.872224e-02,False,Naive
6,Naive,Persistence-Enhanced Log-Return Transformer,8.491742e-35,3.736366e-33,True,Naive
7,Naive,Chronos-Bolt-Tiny,1.196622e-07,2.632568e-06,True,Naive
8,Naive,TimesFM,2.612603e-05,4.441425e-04,True,Naive
9,Simple Exponential Smoothing — Rolling One-Step,Additive-Trend Exponential Smoothing,1.927922e-02,1.927922e-01,False,Simple Exponential Smoothing — Rolling One-Step


In [9]:
pd.read_csv(ROOT/'results'/'bitcoin_transparency_auditability_rubric.csv')

,Model,Mechanism Transparency,Artifact Reproducibility,Deterministic Behaviour,Implementation Simplicity,Failure Detectability,External Independence,Transparency and Auditability Score
0,Naive,100,100,100,100,100,100,100.000000
1,Simple Exponential Smoothing — Rolling One-Step,95,100,100,90,95,100,96.666667
2,Additive-Trend Exponential Smoothing,92,100,100,88,92,100,95.333333
3,ARIMA Rolling One-Step,85,100,100,82,90,100,92.833333
4,7-Day Moving Average,98,100,100,98,98,100,99.000000
5,Prophet — 30-Day Periodic Refit,75,100,90,72,82,100,86.500000
6,Persistence-Enhanced Log-Return LSTM,50,100,90,55,70,100,77.500000
7,Persistence-Enhanced Log-Return Transformer,45,100,100,45,75,100,77.500000
8,Chronos-Bolt-Tiny,35,100,90,60,82,45,68.666667
9,TimesFM,30,100,90,50,78,45,65.500000


In [10]:
pd.read_csv(ROOT/'results'/'bitcoin_trustworthiness_components_v2.csv').sort_values('Exploratory Composite — Missing Evidence Penalised',ascending=False)

,Model,Point Forecast Accuracy,Regime-Conditional Robustness,Temporal Stability,Uncertainty Calibration,Transparency and Auditability,Uncertainty Evidence Type,Exploratory Composite — Missing Evidence Penalised,Exploratory Composite — Evidence Available
4,Naive,100.000000,98.672507,100.000000,82.115928,100.000000,Validation-Residual Empirical Interval,97.051891,97.051891
8,Simple Exponential Smoothing — Rolling One-Step,99.886479,99.070597,99.913332,83.647502,96.666667,Validation-Residual Empirical Interval,96.970846,96.970846
1,ARIMA Rolling One-Step,99.320684,100.000000,98.988513,82.822809,92.833333,Validation-Residual Empirical Interval,96.266697,96.266697
2,Additive-Trend Exponential Smoothing,99.034173,98.611815,99.193747,82.822809,95.333333,Validation-Residual Empirical Interval,96.179827,96.179827
3,Chronos-Bolt-Tiny,92.959750,96.352630,93.750212,98.091423,68.666667,Training-Only CQR-Adjusted Foundation Quantiles,92.136861,92.136861
9,TimesFM,96.332263,97.639143,95.992177,69.509896,65.500000,Training-Only CQR-Adjusted Foundation Quantiles,89.419041,89.419041
5,Persistence-Enhanced Log-Return LSTM,98.539868,97.110309,98.407655,NaN,77.500000,Unavailable,81.342547,95.697114
0,7-Day Moving Average,61.795627,77.559328,61.660333,83.411876,99.000000,Validation-Residual Empirical Interval,71.884183,71.884183
6,Persistence-Enhanced Log-Return Transformer,72.414295,77.609870,71.154571,51.837889,77.500000,Validation-Residual Empirical Interval,70.623575,70.623575
7,Prophet — 30-Day Periodic Refit,17.193180,25.724475,17.036188,NaN,86.500000,Unavailable,23.219746,27.317348


## Limitations and findings
The last UTC date is partial through 01:57; model contexts and update policies differ; Prophet updates only every 30 targets; foundation pretraining overlap is unresolved; uncertainty methods are heterogeneous; and composite weights are subjective. Under the frozen rolling one-day protocol, simple persistence remains exceptionally strong and complexity does not guarantee improved point accuracy.